# Round 8 — Reference-only geometry conditioning

**One independent, bounded feature experiment.** Twelve new CPU readouts, six saved controls, no encoder calls or downloads. The repeatedly inspected 881-comment development cohort is not a fresh holdout or a Kaggle score.

This round was frozen together with its companion before either result was available. Neither uses the other round’s outcomes.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "configs/conditioned_geometry_features.json").is_file())
from scripts.run_conditioned_geometry_features import figures, write_dashboard, json_hash
from jigsaw_rules.runtime import digest
config = json.loads((ROOT / "configs/conditioned_geometry_features.json").read_text())
print("Primary:", config["primary"])
print("New fits:", config["new_fits"], "| Saved controls:", config["cached_control_readouts"])


Primary: dual
New fits: 12 | Saved controls: 6


## 1. Measured starting point

Round 7’s combined pair features reached 0.720881 macro AUC, below raw basic geometry (0.723068). Its primary failed. That result is preserved, not presented as an accepted improvement. The table below is loaded from its hash-pinned report.

In [2]:
previous_path = ROOT / "reports/matched_support_features/results.json"
assert digest(previous_path) == config["round7_results_sha256"]
previous = json.loads(previous_path.read_text())
display(pd.DataFrame(previous["pooled_metrics"]))
print("Prior decision:", previous["decision"])


,variant,macro_auc,pooled_auc,ranked_pooled_auc
0,qwen_raw,0.719893,0.739666,0.738959
1,answer_only,0.719893,0.738680,0.738959
2,frozen_basic,0.723068,0.733852,0.737090
3,paired_global,0.720518,0.734005,0.736241
4,paired_local,0.722379,0.735450,0.736958
5,paired_all,0.720881,0.735856,0.736589
6,repaired_all,0.718273,0.733251,0.733773
7,orientation_all,0.721683,0.734279,0.735956
8,unnormalized_all,0.721144,0.735128,0.736415


Prior decision: DO_NOT_PROMOTE_PRIMARY


## 2. Registered construction and evidence integrity

Centering and training-reference covariance fitting precede nine same-rule class-similarity summaries per transform. The primary combines four-direction removal and regularized low-rank whitening. Random-axis and reference-label permutation controls test the mechanism. The representation is frozen; there is no new encoder training.

The terminal helper computes first and records the immutable result. This notebook verifies and displays it; opening or replaying it never silently trains models.

In [3]:
result_path = ROOT / "reports/conditioned_geometry_features/results.json"
if not result_path.is_file():
    raise RuntimeError("Run the corresponding bounded helper first.")
result = json.loads(result_path.read_text())
marker_path = ROOT / "runs/conditioned_geometry_features" / result["run_id"] / "finished.json"
marker = json.loads(marker_path.read_text())
assert marker["public_hashes"]["results.json"] == digest(result_path)
assert marker["identity"] == result["identity"]
assert json_hash(result["identity"])[:20] == result["run_id"]
CHARTS = figures(result)
print("Run:", result["run_id"], "| Decision:", result["decision"])
print("Control parity:", result["control_design_parity"])


Run: eab42d2b77ef0e2ac8e0 | Decision: DO_NOT_PROMOTE_PRIMARY
Control parity: True


## 3. Policy performance and uncertainty

Both policies matter; a mean can conceal regressions. Bands cover the predeclared comparisons in this round conditional on fixed predictions, not every adaptive experiment in the project or model-refit variability.

In [4]:
display(pd.DataFrame(result["metrics"]))
CHARTS[0].show(renderer="plotly_mimetype")
CHARTS[1].show(renderer="plotly_mimetype")


,fold,policy,variant,auc,brier,log_loss
0,0,"No Advertising: Spam, referral links, unsolici...",qwen_raw,0.679254,0.250238,0.760003
1,1,No legal advice: Do not offer or request legal...,qwen_raw,0.760533,0.221528,0.709879
2,0,"No Advertising: Spam, referral links, unsolici...",answer_only,0.679254,0.260747,0.828707
3,1,No legal advice: Do not offer or request legal...,answer_only,0.760533,0.228357,0.767258
4,0,"No Advertising: Spam, referral links, unsolici...",frozen_basic,0.693097,0.254469,0.827292
5,1,No legal advice: Do not offer or request legal...,frozen_basic,0.753038,0.234327,0.821515
6,0,"No Advertising: Spam, referral links, unsolici...",centered,0.694216,0.254651,0.832126
7,1,No legal advice: Do not offer or request legal...,centered,0.751874,0.236653,0.837628
8,0,"No Advertising: Spam, referral links, unsolici...",deflated,0.674664,0.265643,0.879095
9,1,No legal advice: Do not offer or request legal...,deflated,0.749858,0.235387,0.827622


## 4. Mechanism controls and family removals

A primary gain against raw Qwen alone is insufficient. It must also beat the answer-only readout, raw basic features and the mechanism controls. Full-minus-family comparisons quantify the separate contributions; no secondary winner replaces the registered primary.

In [5]:
display(pd.DataFrame(result["comparisons"]))
CHARTS[2].show(renderer="plotly_mimetype")
CHARTS[3].show(renderer="plotly_mimetype")


,candidate,reference,comparison,delta_auc,simultaneous_low,simultaneous_high
0,centered,frozen_basic,centered vs basic geometry,-0.000022,-0.018801,0.018756
1,deflated,frozen_basic,deflated vs basic geometry,-0.010806,-0.029585,0.007972
2,whitened,frozen_basic,whitened vs basic geometry,-0.000928,-0.019706,0.017850
3,dual,frozen_basic,dual vs basic geometry,-0.007551,-0.026330,0.011227
4,random_dual,frozen_basic,random_dual vs basic geometry,-0.000730,-0.019508,0.018048
5,label_null_dual,frozen_basic,label_null_dual vs basic geometry,-0.003625,-0.022404,0.015153
6,dual,qwen_raw,Primary vs raw Qwen,-0.004377,-0.023155,0.014401
7,dual,answer_only,Primary vs answer-only,-0.004377,-0.023155,0.014401
8,dual,random_dual,Estimated axes vs spectrum-matched random axes,-0.006821,-0.025599,0.011957
9,dual,label_null_dual,Supplied labels vs permuted labels,-0.003926,-0.022704,0.014852


## 5. Reference coverage and diagnostics

The common cosine and spectrum summaries describe reference geometry, not accuracy. Better isotropy is not itself a promotion gate. No query enters the transform fit.

In [6]:
display(pd.DataFrame(result["diagnostics"]).query("inner_fold == 'outer_query'"))
CHARTS[4].show(renderer="plotly_mimetype")
CHARTS[5].show(renderer="plotly_mimetype")


,fold,inner_fold,self_text_overlap,references,dimensions,spectral_rank,removed_directions,shrinkage,mean_vector_norm,captured_variance_fraction,top_removed_variance_fraction,minimum_relative_whitening_gain,mode,raw_reference_mean_cosine,transformed_reference_mean_cosine,query_rows,query_used_in_transform_fit
15,0,outer_query,0,629,2560,32,4,0.2,0.983156,0.873480,0.677902,0.014421,centered,0.966543,0.025937,234,False
16,0,outer_query,0,629,2560,32,4,0.2,0.983156,0.873480,0.677902,0.014421,deflated,0.966543,-0.000988,234,False
17,0,outer_query,0,629,2560,32,4,0.2,0.983156,0.873480,0.677902,0.014421,whitened,0.966543,-0.001376,234,False
18,0,outer_query,0,629,2560,32,4,0.2,0.983156,0.873480,0.677902,0.014421,random_deflated,0.966543,0.025962,234,False
19,0,outer_query,0,629,2560,32,4,0.2,0.983156,0.873480,0.677902,0.014421,random_whitened,0.966543,0.025734,234,False
35,1,outer_query,0,366,2560,32,4,0.2,0.972127,0.903982,0.744351,0.013388,centered,0.944881,0.020621,647,False
36,1,outer_query,0,366,2560,32,4,0.2,0.972127,0.903982,0.744351,0.013388,deflated,0.944881,-0.002342,647,False
37,1,outer_query,0,366,2560,32,4,0.2,0.972127,0.903982,0.744351,0.013388,whitened,0.944881,-0.002657,647,False
38,1,outer_query,0,366,2560,32,4,0.2,0.972127,0.903982,0.744351,0.013388,random_deflated,0.944881,0.020618,647,False
39,1,outer_query,0,366,2560,32,4,0.2,0.972127,0.903982,0.744351,0.013388,random_whitened,0.944881,0.020754,647,False


## 6. Fitted associations and probability quality

Standardized coefficients are descriptive, not causal importance. Matched additions/removals and policy stability are the evidence of feature value. Brier score and log loss may worsen even when AUC increases.

In [7]:
display(pd.DataFrame(result["pooled_metrics"]))
CHARTS[6].show(renderer="plotly_mimetype")
CHARTS[7].show(renderer="plotly_mimetype")


,variant,macro_auc,pooled_auc,ranked_pooled_auc
0,qwen_raw,0.719893,0.739666,0.738959
1,answer_only,0.719893,0.738680,0.738959
2,frozen_basic,0.723068,0.733852,0.737090
3,centered,0.723045,0.732776,0.736526
4,deflated,0.712261,0.731647,0.729849
5,whitened,0.722140,0.732096,0.733957
6,dual,0.715516,0.729907,0.729047
7,random_dual,0.722337,0.732581,0.736067
8,label_null_dual,0.719442,0.730371,0.730244


## 7. Fixed decision, limitations, and handoff

Primary: `dual`. Require +0.003 macro AUC, a positive simultaneous lower bound, and no per-policy regression against every registered comparator. Ranked-pooled AUC cannot decline versus raw Qwen. A pass is eligibility for further validation only.

The cached adapted **training answer margins remain in-sample** on support labels. Cross-fitting new features does not repair that limitation. These are exploratory development experiments, not independent evidence of a leaderboard gain.

See the accompanying methodology document for equations, research sources, controls, and applicability limits.

In [8]:
for requirement in result["primary_requirements"]:
    display(pd.DataFrame([requirement["contrast"]]))
    print(requirement["reference"], requirement["per_policy_delta"], requirement["passed"])
print("Decision:", result["decision"])
for limitation in result["limitations"]:
    print(limitation)
print("Dashboard:", write_dashboard(ROOT, result))
print("No automatic GPU work, Git push, or model promotion.")


,candidate,reference,comparison,delta_auc,simultaneous_low,simultaneous_high
0,dual,qwen_raw,Primary vs raw Qwen,-0.004377,-0.023155,0.014401


qwen_raw [0.007350746268656727, -0.016105359973386024] False


,candidate,reference,comparison,delta_auc,simultaneous_low,simultaneous_high
0,dual,answer_only,Primary vs answer-only,-0.004377,-0.023155,0.014401


answer_only [0.007350746268656727, -0.016105359973386024] False


,candidate,reference,comparison,delta_auc,simultaneous_low,simultaneous_high
0,dual,frozen_basic,dual vs basic geometry,-0.007551,-0.02633,0.011227


...

,candidate,reference,comparison,delta_auc,simultaneous_low,simultaneous_high
0,dual,random_dual,Estimated axes vs spectrum-matched random axes,-0.006821,-0.025599,0.011957


random_dual [-0.006343283582089465, -0.007299270072992692] False


,candidate,reference,comparison,delta_auc,simultaneous_low,simultaneous_high
0,dual,label_null_dual,Supplied labels vs permuted labels,-0.003926,-0.022704,0.014852


label_null_dual [-0.00970149253731345, 0.0018492788790824832] False
Decision: DO_NOT_PROMOTE_PRIMARY
Exploratory follow-up on repeatedly inspected development data, not a Kaggle score.
Both new rounds were fixed before either result; neither selects the other round.
Cross-fitting excludes each held training group from every fitted reference statistic.
The retained adapted training answer margin is in-sample on supplied support labels.
Cross-fitting new features does not make that answer margin out-of-fold.
Inner reference pools are smaller than the outer inference pool.
Conditional intervals cover this round only, not all adaptive project choices.
A screen pass is eligibility for new validation, never automatic GPU authorization.
Whitening/deflation can remove useful signal; isotropy is a diagnostic, not a target.
Random axes share the estimated spectrum and rank, not equal query row geometry.
Low-rank regularized whitening retains residual directions; it is not full whitening.


Dashboard: /home/sagemaker-user/projects/jigsaw-rule-classifier/reports/conditioned_geometry_features/dashboard.html
No automatic GPU work, Git push, or model promotion.
